# Apache Polaris — Reset Principal

## Concept
Scoped teardown for a single principal and its associated role wiring —
distinct from `polaris_clean_all.ipynb`, which wipes the entire instance.
This notebook resets exactly one principal's footprint:

```
1. GET  Principal            <PRINCIPAL_NAME>
2. GET  Principal Role        <PRINCIPAL_NAME>_role
3. GET  Catalog                my_<PRINCIPAL_NAME>_catalog
4. GET  Catalog Roles           (all roles defined on that catalog)
5. DELETE Catalog Roles         (skip the built-in catalog_admin)
6. DELETE Principal Role
7. DELETE Principal
8. VERIFY                      re-GET everything above, expect 404 / absence
```

**The catalog itself is intentionally left in place** — only the principal,
its principal-role, and the catalog's non-default catalog-roles are deleted.
If you also want the catalog gone, run `admin/polaris_clean_all.ipynb`
separately (or extend this notebook — not in scope here).

Built on `src/polaris_rest.PolarisREST` (the shared REST client) rather than
hand-rolled `requests` calls, per the reusable-client convention.

## Safety
LOCAL DEVICE ONLY, same rationale as `polaris_clean_all.ipynb` and
`polaris_rotate_credential.ipynb`: hardcoded to the local OrbStack host, with
a host-guard assert, never wired to `init_env()` / shared config. It deletes
real principals/roles — do not point it at `dev` or `prod`.

## How to run
1. Set `PRINCIPAL_NAME` in the parameters cell below.
2. **Read the GET/preview cell outputs (steps 1–4) before proceeding** — this
   notebook is destructive from step 5 onward.
3. Restart & Run All only against a local instance you intend to modify.


In [1]:
# ============================================================
# Prerequisites:  uv add requests
# ============================================================
import subprocess, sys, os

REQUIRED = ['requests']

def install(pkg):
    result = subprocess.run(['uv', 'add', pkg], capture_output=True, text=True, cwd=os.getcwd())
    if result.returncode == 0:
        return '✅', 'uv'
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg], capture_output=True, text=True)
    return ('✅', 'pip') if result.returncode == 0 else ('❌', 'failed')

for pkg in REQUIRED:
    status, method = install(pkg)
    print(f'  {status} {pkg} ({method})')
print('\n⚠️  Restart kernel after first install.')

  ✅ requests (uv)
  ✅ pandas (uv)

⚠️  Restart kernel after first install.


In [1]:
# ============================================================
# Setup — hardcoded local device only (see Safety note above)
# ============================================================
import sys, pathlib
from IPython.display import display

_ROOT = pathlib.Path.cwd()
while not (_ROOT / "src").is_dir() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
sys.path.insert(0, str(_ROOT / "src"))
from polaris_rest import PolarisREST

POLARIS_URL = 'http://192.168.139.2:8181'
REALM       = 'POLARIS'
ROOT        = 'root'
ROOT_SECRET = 'polaris-secret'

# ── SAFETY GUARD: LOCAL DEVICE ONLY ─────────────────────────────────
# This notebook deletes a real principal + role wiring. It must never touch a
# shared company environment. Hard-fail on non-local hosts.
assert any(t in POLARIS_URL for t in ("127.0.0.1", "localhost", "192.168.139.2")), \
    f"reset_principal is LOCAL ONLY — refusing non-local host: {POLARIS_URL}"

pc = PolarisREST(POLARIS_URL, REALM)
r = pc.get_token(ROOT, ROOT_SECRET)
pc.token = r.json()['access_token']
print('✅ Connected as root')

✅ Connected as root


In [2]:
# ============================================================
# Parameters — set the target principal before running
# ============================================================
PRINCIPAL_NAME = 'REPLACE_ME'   # e.g. 'alice'

assert PRINCIPAL_NAME != 'REPLACE_ME', \
    'Set PRINCIPAL_NAME to a real principal before running this notebook.'

PRINCIPAL_ROLE = f'{PRINCIPAL_NAME}_role'
CATALOG        = f'my_{PRINCIPAL_NAME}_catalog'

print(f'PRINCIPAL_NAME  = {PRINCIPAL_NAME}')
print(f'PRINCIPAL_ROLE  = {PRINCIPAL_ROLE}')
print(f'CATALOG         = {CATALOG}')

AssertionError: Set PRINCIPAL_NAME to a real principal before running this notebook.

In [5]:
# ============================================================
# Step 1 — GET Principal details
# ============================================================
r = pc.get_principal(PRINCIPAL_NAME)
principal_exists = r.status_code == 200
if principal_exists:
    print(f'✅ Principal found: {PRINCIPAL_NAME}')
    display(r.json())
else:
    print(f'⚠️  Principal not found ({r.status_code}): {PRINCIPAL_NAME} — nothing to reset here, continuing to check the rest.')

⚠️  Principal not found (404): my-principal — nothing to reset here, continuing to check the rest.


In [ ]:
# ============================================================
# Step 2 — GET Principal Role
# ============================================================
r = pc.get_principal_role(PRINCIPAL_ROLE)
principal_role_exists = r.status_code == 200
if principal_role_exists:
    print(f'✅ Principal role found: {PRINCIPAL_ROLE}')
    display(r.json())
else:
    print(f'⚠️  Principal role not found ({r.status_code}): {PRINCIPAL_ROLE}')

In [ ]:
# ============================================================
# Step 3 — GET Catalog
# ============================================================
r = pc.get_catalog(CATALOG)
catalog_exists = r.status_code == 200
if catalog_exists:
    print(f'✅ Catalog found: {CATALOG}')
    display(r.json())
else:
    print(f'⚠️  Catalog not found ({r.status_code}): {CATALOG} — skipping catalog-role steps.')

In [ ]:
# ============================================================
# Step 4 — GET Catalog Roles (build the delete list, skip catalog_admin)
# ============================================================
catalog_roles = []
delete_targets = []
if catalog_exists:
    r = pc.list_catalog_roles(CATALOG)
    catalog_roles = r.json().get('roles', []) if r.status_code == 200 else []
    for cr in catalog_roles:
        tag = ' (system — will not delete)' if cr['name'] == 'catalog_admin' else ''
        print(f'  🔑 catalog-role: {cr["name"]}{tag}')
        if cr['name'] != 'catalog_admin':
            delete_targets.append(cr['name'])

print(f'\nWill delete {len(delete_targets)} catalog-role(s): {delete_targets}')

In [ ]:
# ============================================================
# Step 5 — DELETE Catalog Roles (non-default only)
# ============================================================
for crname in delete_targets:
    r = pc.delete_catalog_role(CATALOG, crname)
    icon = '✅' if r.status_code in (200, 204, 404) else '⚠️ '
    print(f'  {icon} [{r.status_code}] catalog-role: {CATALOG}/{crname}')

In [ ]:
# ============================================================
# Step 6 — DELETE Principal Role
# ============================================================
r = pc.delete_principal_role(PRINCIPAL_ROLE)
icon = '✅' if r.status_code in (200, 204, 404) else '⚠️ '
print(f'{icon} [{r.status_code}] principal-role: {PRINCIPAL_ROLE}')

In [ ]:
# ============================================================
# Step 7 — DELETE Principal
# ============================================================
r = pc.delete_principal(PRINCIPAL_NAME)
icon = '✅' if r.status_code in (200, 204, 404) else '⚠️ '
print(f'{icon} [{r.status_code}] principal: {PRINCIPAL_NAME}')

In [ ]:
# ============================================================
# Step 8 — VERIFY everything deleted above is actually gone
# ============================================================
print('=== Verification ===\n')

checks = {}

r = pc.get_principal(PRINCIPAL_NAME)
checks[f'principal: {PRINCIPAL_NAME}'] = r.status_code == 404

r = pc.get_principal_role(PRINCIPAL_ROLE)
checks[f'principal-role: {PRINCIPAL_ROLE}'] = r.status_code == 404

if catalog_exists:
    r = pc.list_catalog_roles(CATALOG)
    remaining = [cr['name'] for cr in r.json().get('roles', [])] if r.status_code == 200 else []
    remaining_non_default = [n for n in remaining if n != 'catalog_admin']
    checks[f'catalog-roles cleared on {CATALOG} (remaining={remaining_non_default})'] = len(remaining_non_default) == 0

all_clean = True
for label, ok in checks.items():
    icon = '✅' if ok else '⚠️ '
    if not ok:
        all_clean = False
    print(f'  {icon} {label}')

print()
if all_clean:
    print(f'✅ Reset complete — {PRINCIPAL_NAME} and its role wiring are gone. Catalog {CATALOG} was left in place by design.')
else:
    print('⚠️  Some resources remain — check above.')